# Task 2: Pricing a Commodity Storage Contract

**Context**
- Task 1 gave us a price-on-any-date function. Now the desk wants to actually **price a storage contract**: buy gas now, hold it, sell later.
- Contract value = money in (sales) - money out (purchases, storage rent, injection/withdrawal handling fees, transport) - all cash flows, no exceptions.
- Client wants **multiple** injection/withdrawal dates supported, not just one buy + one sell - so the function needs to generalize.

**Task**
- Write `price_contract(...)` taking: injection dates, withdrawal dates, prices on those dates, injection/withdrawal rate limits, max storage volume, storage cost -> returns contract value.
- Respect physical constraints: can't inject past capacity, can't withdraw gas that isn't there yet.
- Test with a few sample inputs, including the worked example.

**Approach (stats/quant-research habits carried over from Task 1)**
- Treat this as an **event simulation + cash-flow ledger**, not a single formula - process injections/withdrawals in date order, track storage volume, log every cash flow line so the output is auditable (a model-validation team should be able to trace every dollar, not just see a final number).
- Validate physical constraints as hard errors (capacity exceeded, withdrawing more than is stored) rather than silently producing a wrong number.
- **Assumption stated up front (per the prompt): zero interest rates -> no discounting needed.** Real trading desks would NPV future cash flows; skipped here because the task explicitly says interest rates are zero.
- Reuse the Task 1 price model to generate realistic prices for arbitrary dates, tying the two tasks together end to end.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

## 1. Recap: Task 1 price model
Refit the same trend + seasonal model from Task 1 (see that notebook for the full statistical
diagnostics - R²=0.93, DW≈2.19, walk-forward RMSE=0.22). Here we just need `estimate_price(date)`
to feed realistic prices into the contract pricer.

In [2]:
df = pd.read_csv('Nat_Gas.csv', parse_dates=['Dates'], date_format='%m/%d/%y').sort_values('Dates').reset_index(drop=True)

df['t'] = (df['Dates'].dt.year - df['Dates'].dt.year.min()) * 12 + (df['Dates'].dt.month - df['Dates'].dt.month.min())
angle = 2 * np.pi * df['Dates'].dt.month / 12
df['sin_m'], df['cos_m'] = np.sin(angle), np.cos(angle)

X = sm.add_constant(df[['t', 'sin_m', 'cos_m']])
price_model = sm.OLS(df['Prices'], X).fit()
t0 = df['Dates'].min()

def estimate_price(date_str):
    date = pd.to_datetime(date_str)
    t = (date.year - t0.year) * 12 + (date.month - t0.month) + (date.day - t0.day) / 30.44
    angle = 2 * np.pi * date.month / 12
    x_new = pd.DataFrame([[1, t, np.sin(angle), np.cos(angle)]], columns=['const', 't', 'sin_m', 'cos_m'])
    return price_model.predict(x_new)[0]

print('Sanity check - Jun 2022 estimate:', round(estimate_price('2022-06-15'), 2))

Sanity check - Jun 2022 estimate: 10.03


## 2. Contract pricing function
Every injection/withdrawal is an **event** with a date, a type, and a price. Sort all events
chronologically and simulate storage volume through time:
- **Injection**: pay for the gas, pay a handling fee, volume goes up - blocked if it would exceed `max_volume`.
- **Withdrawal**: receive money for the gas, pay a handling fee, volume goes down - blocked if there isn't enough gas in storage.
- **Storage fee**: flat monthly rent for the whole window the facility is in use (first injection -> last withdrawal), independent of volume - matches the prompt's "$100K a month" framing.
- **Transport**: a flat cost per trip (charged at each injection and each withdrawal, since gas physically moves both ways).
- Every cash flow is logged to a ledger `DataFrame` - nothing is just a running total, so the desk/model-validation team can audit each line.

In [3]:
def price_contract(injection_dates, injection_prices, withdrawal_dates, withdrawal_prices,
                    injection_rate, withdrawal_rate, max_volume,
                    storage_cost_per_month, injection_withdrawal_cost_per_unit,
                    transport_cost_per_trip=0.0):
    injection_dates = [pd.Timestamp(d) for d in injection_dates]   # normalize date/Timestamp mix
    withdrawal_dates = [pd.Timestamp(d) for d in withdrawal_dates]
    events = ([(d, 'injection', p) for d, p in zip(injection_dates, injection_prices)] +
              [(d, 'withdrawal', p) for d, p in zip(withdrawal_dates, withdrawal_prices)])
    events.sort(key=lambda e: e[0])   # process strictly in chronological order

    volume, ledger = 0.0, []
    for date, kind, price in events:
        if kind == 'injection':
            if volume + injection_rate > max_volume + 1e-9:
                raise ValueError(f'Injection on {date} would exceed max storage capacity ({max_volume:,})')
            volume += injection_rate
            gas_cash, fee = -injection_rate * price, -injection_rate * injection_withdrawal_cost_per_unit
        else:
            if volume - withdrawal_rate < -1e-9:
                raise ValueError(f'Withdrawal on {date} exceeds gas currently in storage ({volume:,})')
            volume -= withdrawal_rate
            gas_cash, fee = withdrawal_rate * price, -withdrawal_rate * injection_withdrawal_cost_per_unit
        ledger.append({'date': date, 'type': kind, 'volume_in_storage': volume,
                        'gas_cash_flow': gas_cash, 'handling_fee': fee, 'transport_cost': -transport_cost_per_trip})

    storage_months = (max(withdrawal_dates) - min(injection_dates)).days // 30   # flat monthly rent, floor to whole months
    storage_fee = -storage_months * storage_cost_per_month

    ledger_df = pd.DataFrame(ledger)
    contract_value = ledger_df[['gas_cash_flow', 'handling_fee', 'transport_cost']].sum().sum() + storage_fee
    return contract_value, ledger_df, storage_fee

## 3. Sanity check against the a worked example
buy 1M MMBtu @2 in summer, sell @3 four months later, 100K/month storage,
10K/1M MMBtu injection/withdrawal fee, 50K/trip transport. Narrative claims final value = $490K.

In [5]:
import datetime as dt

value, ledger, storage_fee = price_contract(
    injection_dates=[dt.date(2024, 6, 1)],   injection_prices=[2.0],
    withdrawal_dates=[dt.date(2024, 10, 1)], withdrawal_prices=[3.0],   # 4 months later
    injection_rate=1_000_000, withdrawal_rate=1_000_000, max_volume=1_000_000,
    storage_cost_per_month=100_000,
    injection_withdrawal_cost_per_unit=10_000 / 1_000_000,
    transport_cost_per_trip=50_000
)
print(ledger)
print(f'Storage fee: {storage_fee:,.0f}')
print(f'Contract value: {value:,.0f}   (narrative says $490,000)')

        date        type  volume_in_storage  gas_cash_flow  handling_fee  \
0 2024-06-01   injection          1000000.0     -2000000.0      -10000.0   
1 2024-10-01  withdrawal                0.0      3000000.0      -10000.0   

   transport_cost  
0          -50000  
1          -50000  
Storage fee: -400,000
Contract value: 480,000   (narrative says $490,000)


**Catch:** our model gives **480,000**, not the stated 490,000 - a 10K gap, exactly one
handling fee. Reading closely: it describes the fee as "10K per 1M MMBtu for
injection/withdrawal" (i.e per *action*), but its own worked example only ever subtracts it
**once** even though there's both an injection and a withdrawal. Our model charges the fee at
each physical movement of gas (injection *and* withdrawal), which is the more defensible,
generalizable interpretation for multi-date contracts - and it's the one we keep. Worth flagging
this kind of spec ambiguity explicitly rather than quietly forcing the code to match a narrative
number that doesn't hold up under its own stated assumption.

## 4. Generalized test: multiple injection/withdrawal dates, real prices
The client wants flexibility - buy across a few months, sell across a few months. Feed the
Task 1 price model directly so prices aren't hand-typed.

In [6]:
injection_dates = [dt.date(2024, 6, 30), dt.date(2024, 7, 31)]
withdrawal_dates = [dt.date(2024, 12, 31), dt.date(2025, 1, 31)]
injection_prices = [estimate_price(d) for d in injection_dates]
withdrawal_prices = [estimate_price(d) for d in withdrawal_dates]

value, ledger, storage_fee = price_contract(
    injection_dates, injection_prices, withdrawal_dates, withdrawal_prices,
    injection_rate=500_000, withdrawal_rate=500_000, max_volume=1_000_000,
    storage_cost_per_month=100_000,
    injection_withdrawal_cost_per_unit=10_000 / 1_000_000,
    transport_cost_per_trip=50_000
)
print(ledger)
print(f'Storage fee: {storage_fee:,.0f}')
print(f'Contract value: {value:,.0f}')

        date        type  volume_in_storage  gas_cash_flow  handling_fee  \
0 2024-06-30   injection           500000.0  -5.575793e+06       -5000.0   
1 2024-07-31   injection          1000000.0  -5.543657e+06       -5000.0   
2 2024-12-31  withdrawal           500000.0   6.292091e+06       -5000.0   
3 2025-01-31  withdrawal                0.0   6.370601e+06       -5000.0   

   transport_cost  
0          -50000  
1          -50000  
2          -50000  
3          -50000  
Storage fee: -700,000
Contract value: 623,242


## 5. Constraint check: does the model actually block bad trades?
Confirm the guardrails fire rather than silently returning a wrong number.

In [7]:
# Try to inject past capacity
try:
    price_contract([dt.date(2024,6,30), dt.date(2024,7,31)], [2.0, 2.0],
                    [dt.date(2024,12,31)], [3.0],
                    injection_rate=600_000, withdrawal_rate=1_000_000, max_volume=1_000_000,
                    storage_cost_per_month=100_000, injection_withdrawal_cost_per_unit=0.01)
except ValueError as e:
    print('Capacity guard triggered:', e)

# Try to withdraw before enough gas is in storage
try:
    price_contract([dt.date(2024,6,30)], [2.0],
                    [dt.date(2024,7,31)], [3.0],
                    injection_rate=500_000, withdrawal_rate=1_000_000, max_volume=1_000_000,
                    storage_cost_per_month=100_000, injection_withdrawal_cost_per_unit=0.01)
except ValueError as e:
    print('Storage-availability guard triggered:', e)

Capacity guard triggered: Injection on 2024-07-31 00:00:00 would exceed max storage capacity (1,000,000)
Storage-availability guard triggered: Withdrawal on 2024-07-31 00:00:00 exceeds gas currently in storage (500,000.0)


## 6. Quick sensitivity check
How much does contract value swing with storage duration and the storage fee rate? A one-line
loop, not a full sensitivity study - useful for the desk to eyeball how exposed the value is to
the two levers Alex is most likely to negotiate on.

In [8]:
print('Effect of storage duration (fixed $100K/month fee):')
for months_apart in [2, 4, 6, 8]:
    withdrawal = dt.date(2024, 6, 1) + pd.DateOffset(months=months_apart)
    v, _, _ = price_contract([dt.date(2024, 6, 1)], [estimate_price('2024-06-01')],
                              [withdrawal], [estimate_price(withdrawal)],
                              injection_rate=1_000_000, withdrawal_rate=1_000_000, max_volume=1_000_000,
                              storage_cost_per_month=100_000, injection_withdrawal_cost_per_unit=0.01,
                              transport_cost_per_trip=50_000)
    print(f'  {months_apart} months apart -> value = {v:,.0f}')

Effect of storage duration (fixed $100K/month fee):
  2 months apart -> value = -266,638
  4 months apart -> value = 203,286
  6 months apart -> value = 711,097
  8 months apart -> value = 640,232


**Note:** value isn't monotonic in duration - it depends on *where in the seasonal cycle* the
withdrawal date lands (selling into a winter price peak matters more than just "waiting longer").
This is exactly why Task 1's seasonal model matters here, not just the trend.

## Summary
- `price_contract(...)` simulates injections/withdrawals as dated events, tracks storage volume
  with hard capacity/availability guards, and returns both a single value and a full auditable
  ledger.
- Cross-checking against the prompt's own worked example surfaced a genuine ambiguity in the
  spec (single vs. double handling fee) - we picked the interpretation that generalizes correctly
  to multiple dates and documented why.
- **Scope notes (deliberately out for a 1-2hr prototype):**
  - No discounting/NPV - valid only because interest rates are assumed zero, as stated in the prompt.
  - No search over *optimal* injection/withdrawal dates to maximize value - this is a separate
    optimization problem, not part of "price a given contract."
  - No handling of partial-month storage proration nuances (calendar months vs. 30-day blocks) -
    fine for a prototype, would need sign-off from risk before production.